# 🔬 Notebook 4 — Use Your Trained Model on New DNA

## The final question

> **Can the model we trained on K562 CTCF recognize CTCF-associated DNA from another human cell type?**

Both groups use this notebook:

```mermaid
flowchart TD
    A["Group A<br/>DNABERT"] --> C["Notebook 4"]
    B["Group B<br/>Custom Transformer"] --> C

    C --> D["Load saved checkpoint"]
    D --> E["Load ENCSR000AOO<br/>astrocyte test DNA"]
    E --> F["Use SAME tokenizer as training"]
    F --> G["Inference"]
    G --> H["Binding probability"]
    H --> I["External evaluation"]
```

The model was trained on **K562** DNA.  
The external test set comes from **ENCSR000AOO astrocyte CTCF** data.

This is different from ordinary validation because the biological context has changed.

## Training vs. inference

You already trained the model.

Now we perform **inference**.

```mermaid
flowchart LR
    A["TRAINING"] --> B["Predict"]
    B --> C["Calculate loss"]
    C --> D["Calculate gradients"]
    D --> E["Update weights"]

    F["INFERENCE"] --> G["Load trained weights"]
    G --> H["Predict new DNA"]
    H --> I["Do NOT update weights"]
```

During inference:

```python
model.eval()
```

means:

> Put the model in prediction mode.

And:

```python
with torch.no_grad():
```

means:

> We do not need gradients because we are not training.

# 1. Choose your group

Group A trained **DNABERT**.

Group B trained a **custom Transformer**.

Change only this variable:

```python
GROUP = "A"
```

or:

```python
GROUP = "B"
```

For Group B, `CUSTOM_TOKENIZER = "best"` automatically chooses the tokenizer with the highest final validation AUROC.

You can also choose one directly:

```text
single_nucleotide
one_hot
overlap_6mer
nonoverlap_6mer
bpe
```

In [ ]:
# ✏️ CHANGE — choose which trained model you are testing

GROUP = "A"              # "A" = DNABERT, "B" = custom Transformer

CUSTOM_TOKENIZER = "best"   # Used only for Group B

# 2. Paths and toolboxes

The important paths are already filled in for Perlmutter.

### `TEST_DATA_PATH`

The external ENCSR000AOO dataset you prepared.

### `MODEL_PATH`

The original local DNABERT files. Group A needs these to rebuild the DNABERT architecture before loading the student's learned weights.

### `PROJECT_DIR`

Where Notebook 3A / 3B wrote their result folders.

In [ ]:
# ▶️ RUN — imports and paths

import os
import itertools
import json

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer, BertModel

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    precision_recall_curve,
)


PROJECT_DIR = Path(
    os.environ.get("DNA_BOOTCAMP_HOME", ".")
).expanduser().resolve()


TEST_DATA_PATH = Path(
    "/global/cfs/cdirs/m4388/projects/project7/"
    "test_data/ENCSR000AOO_external_ctcf_test.csv"
)


MODEL_PATH = Path(
    "/global/cfs/cdirs/m4388/projects/project7/"
    "models/DNA_bert_6"
)


GROUP_A_RESULTS = (
    PROJECT_DIR
    / "notebook3a_results"
)

GROUP_B_RESULTS = (
    PROJECT_DIR
    / "notebook3b_results"
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Group:", GROUP)
print("External test data:", TEST_DATA_PATH)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# 3. Load the external DNA

We use pandas here because the external dataset naturally has several columns:

```text
id
sequence
true_label
label_name
source
cell_type
```

### Function: `pd.read_csv(...)`

This simply means:

> Read the CSV file and display it as rows and columns.

Before prediction, we check that every DNA sequence:

- is exactly **200 bp**,
- contains only **A/C/G/T**,
- has a true label of `0` or `1`.

In [ ]:
# ▶️ RUN — load and check ENCSR000AOO external data

if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(
        f"External test file was not found:\n{TEST_DATA_PATH}\n\n"
        "Create/copy the ENCSR000AOO external test CSV first."
    )


test_data = pd.read_csv(
    TEST_DATA_PATH
)


# Support either 'true_label' or a simpler 'label' column.
if "true_label" not in test_data.columns:
    if "label" in test_data.columns:
        test_data["true_label"] = test_data["label"]
    else:
        raise ValueError(
            "The CSV needs a true_label column."
        )


test_data["sequence"] = (
    test_data["sequence"]
    .astype(str)
    .str.upper()
)


valid_length = (
    test_data["sequence"]
    .str.len()
    .eq(200)
)


valid_dna = (
    test_data["sequence"]
    .str.fullmatch(r"[ACGT]{200}")
    .fillna(False)
)


valid_label = (
    test_data["true_label"]
    .isin([0, 1])
)


if not (
    valid_length
    & valid_dna
    & valid_label
).all():

    bad_rows = test_data[
        ~(
            valid_length
            & valid_dna
            & valid_label
        )
    ]

    raise ValueError(
        "Some external-test rows are invalid. "
        f"Problem rows: {len(bad_rows)}"
    )


print("External examples:", len(test_data))
print()
print(
    test_data["true_label"]
    .value_counts()
    .sort_index()
)

test_data.head()

## What are we testing?

```mermaid
flowchart LR
    A["Training<br/>K562 CTCF"] --> C["Same prediction task"]
    B["External test<br/>Astrocyte CTCF"] --> C
    C --> D["Binding vs Background"]
```

The model has **not been retrained on the astrocyte dataset**.

That is the point.

We want to test whether what it learned from K562 transfers to this external biological context.

In [ ]:
# ▶️ RUN — look at external class balance

class_counts = (
    test_data["true_label"]
    .map({
        0: "Background",
        1: "Binding",
    })
    .value_counts()
)

class_counts.plot(
    kind="bar"
)

plt.ylabel("Number of external sequences")
plt.title("ENCSR000AOO external test-set balance")
plt.show()

# 4. Hidden helper code

The next hidden cell contains the machinery needed to reload either model family.

Students do **not** need to study every line.

The important idea is:

```mermaid
flowchart LR
    A["Checkpoint"] --> B["Rebuild same architecture"]
    B --> C["Load learned state_dict"]
    C --> D["model.eval()"]
    D --> E["Ready for new DNA"]
```

A **checkpoint** contains the weights learned during training.

The architecture must match the architecture that created those weights.

In [ ]:
# 🔒 HELPER — model definitions, tokenizer reconstruction, and inference


# =========================================================
# General checkpoint loader
# =========================================================

def load_checkpoint(path):
    """
    Load a checkpoint created by our own bootcamp notebooks.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found:\n{path}"
        )

    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        # Compatibility with PyTorch versions that do not
        # have the weights_only argument.
        return torch.load(
            path,
            map_location="cpu",
        )


# =========================================================
# GROUP A — DNABERT
# =========================================================

def kmer_sentence(sequence, k=6):
    """
    Convert one DNA sequence into overlapping k-mers.
    """

    return " ".join(
        sequence[i:i+k]
        for i in range(
            len(sequence) - k + 1
        )
    )


class MaximumDNABertClassifier(nn.Module):
    """
    Same architecture used in Notebook 3A.
    """

    def __init__(
        self,
        model_path,
        dropout,
    ):
        super().__init__()

        self.bert = BertModel.from_pretrained(
            str(model_path),
            local_files_only=True,
            add_pooling_layer=False,
        )

        hidden_size = (
            self.bert.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_size,
            2,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
    ):

        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_vector = (
            output
            .last_hidden_state[:, 0, :]
        )

        return self.classifier(
            self.dropout(
                cls_vector
            )
        )


def find_group_a_checkpoint():
    """
    Prefer the path written into the Notebook 3A summary.
    """

    summary_files = sorted(
        GROUP_A_RESULTS.glob(
            "*_summary.json"
        )
    )

    for summary_path in summary_files:
        try:
            with open(summary_path) as handle:
                summary = json.load(handle)

            if (
                summary.get("family") == "dnabert"
                and summary.get("checkpoint_path")
            ):
                checkpoint_path = Path(
                    summary["checkpoint_path"]
                )

                if checkpoint_path.exists():
                    return checkpoint_path, summary
        except Exception:
            pass

    # Fallback for the standard bootcamp run name.
    fallback = (
        GROUP_A_RESULTS
        / "dnabert_maximum_full_finetune_final_checkpoint.pt"
    )

    if fallback.exists():
        return fallback, None

    raise FileNotFoundError(
        "Could not find a Group A DNABERT checkpoint "
        f"inside {GROUP_A_RESULTS}"
    )


def load_group_a():
    """
    Rebuild DNABERT and load the fine-tuned weights.
    """

    checkpoint_path, summary = (
        find_group_a_checkpoint()
    )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    dropout = float(
        checkpoint.get(
            "dropout",
            0.10,
        )
    )

    model = MaximumDNABertClassifier(
        model_path=MODEL_PATH,
        dropout=dropout,
    )

    model.load_state_dict(
        checkpoint["state_dict"]
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(
        str(MODEL_PATH),
        local_files_only=True,
    )

    information = {
        "group": "A",
        "family": "DNABERT",
        "tokenizer": "overlapping 6-mer",
        "checkpoint": str(checkpoint_path),
        "max_length": int(
            checkpoint.get(
                "max_length",
                256,
            )
        ),
        "internal_summary": summary,
    }

    return model, tokenizer, information


def predict_group_a(
    model,
    tokenizer,
    sequences,
    max_length,
    batch_size=64,
):
    """
    Predict new DNA with the fine-tuned DNABERT model.
    """

    probabilities = []

    for start in range(
        0,
        len(sequences),
        batch_size,
    ):

        batch_sequences = (
            sequences[
                start:start + batch_size
            ]
        )

        dna_as_6mers = [
            kmer_sentence(sequence)
            for sequence in batch_sequences
        ]

        encoded = tokenizer(
            dna_as_6mers,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        with torch.no_grad():

            scores = model(
                encoded["input_ids"].to(
                    DEVICE
                ),
                encoded["attention_mask"].to(
                    DEVICE
                ),
            )

            binding_probability = (
                torch.softmax(
                    scores,
                    dim=1,
                )[:, 1]
            )

        probabilities.extend(
            binding_probability
            .cpu()
            .tolist()
        )

    return probabilities


# =========================================================
# GROUP B — CUSTOM TRANSFORMER
# =========================================================

BASE_ID = {
    "A": 1,
    "C": 2,
    "G": 3,
    "T": 4,
}

ONE_HOT = {
    "A": [1., 0., 0., 0.],
    "C": [0., 1., 0., 0.],
    "G": [0., 0., 1., 0.],
    "T": [0., 0., 0., 1.],
}

ALL_6MERS = [
    "".join(chars)
    for chars in itertools.product(
        "ACGT",
        repeat=6,
    )
]

KMER_ID = {
    token: index + 1
    for index, token
    in enumerate(ALL_6MERS)
}


class RestoredBPE:
    """
    Rebuild the exact BPE tokenizer learned in Notebook 3B.
    """

    def __init__(
        self,
        rules,
        vocab,
    ):

        self.rules = [
            (
                tuple(pair),
                merged,
            )
            for pair, merged
            in rules
        ]

        self.vocab = vocab

    def _apply_rule(
        self,
        tokens,
        pair,
        merged,
    ):

        output = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and (
                    tokens[i],
                    tokens[i+1],
                ) == pair
            ):

                output.append(
                    merged
                )

                i += 2

            else:
                output.append(
                    tokens[i]
                )

                i += 1

        return output

    def tokens(
        self,
        sequence,
    ):

        tokens = list(
            sequence
        )

        for pair, merged in self.rules:

            tokens = self._apply_rule(
                tokens,
                pair,
                merged,
            )

        return tokens

    def encode(
        self,
        sequence,
    ):

        return [
            self.vocab.get(
                token,
                1,
            )
            for token
            in self.tokens(
                sequence
            )
        ]


class DNAClassifier(nn.Module):
    """
    Same readable PyTorch architecture used in Notebook 3B.
    """

    def __init__(
        self,
        input_kind,
        vocab_size,
        max_length,
        d_model,
        nhead,
        layers,
        dropout,
    ):
        super().__init__()

        if input_kind == "one_hot":

            self.input_layer = nn.Linear(
                4,
                d_model,
            )

        else:

            self.input_layer = nn.Embedding(
                vocab_size,
                d_model,
                padding_idx=0,
            )

        self.position = nn.Embedding(
            max_length,
            d_model,
        )

        transformer_layer = (
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.transformer = (
            nn.TransformerEncoder(
                transformer_layer,
                num_layers=layers,
            )
        )

        self.classifier = nn.Linear(
            d_model,
            2,
        )

    def forward(
        self,
        x,
        padding_mask,
    ):

        x = self.input_layer(
            x
        )

        positions = torch.arange(
            x.shape[1],
            device=x.device,
        )

        x = (
            x
            + self.position(
                positions
            )
        )

        x = self.transformer(
            x,
            src_key_padding_mask=padding_mask,
        )

        keep = (
            ~padding_mask
        ).unsqueeze(-1)

        dna_summary = (
            (x * keep).sum(dim=1)
            / keep.sum(dim=1)
        )

        return self.classifier(
            dna_summary
        )


def choose_group_b_tokenizer(
    requested,
):
    """
    Choose a specific tokenizer or the best final-validation AUROC.
    """

    valid_names = [
        "single_nucleotide",
        "one_hot",
        "overlap_6mer",
        "nonoverlap_6mer",
        "bpe",
    ]

    if requested != "best":

        if requested not in valid_names:
            raise ValueError(
                "CUSTOM_TOKENIZER must be 'best' "
                "or one of the five tokenizer names."
            )

        return requested, None


    summary_path = (
        GROUP_B_RESULTS
        / "all_tokenizers_summary.json"
    )

    if not summary_path.exists():
        raise FileNotFoundError(
            "Could not choose the best tokenizer because "
            "all_tokenizers_summary.json was not found."
        )

    with open(summary_path) as handle:
        summaries = json.load(handle)

    def score(row):
        return row.get(
            "final_val_auroc",
            row.get(
                "best_val_auroc",
                -1,
            ),
        )

    best = max(
        summaries,
        key=score,
    )

    return (
        best["tokenizer"],
        best,
    )


def find_group_b_checkpoint(
    tokenizer_name,
):
    """
    Find the reusable checkpoint produced by updated Notebook 3B.
    """

    checkpoint_path = (
        GROUP_B_RESULTS
        / f"{tokenizer_name}_final_checkpoint.pt"
    )

    if not checkpoint_path.exists():

        raise FileNotFoundError(
            f"Group B checkpoint not found:\n{checkpoint_path}\n\n"
            "Run Notebook3B_GroupB_Transformer_HPC_Beginner_v3_Checkpoints.ipynb "
            "to create reusable model checkpoints."
        )

    return checkpoint_path


def build_group_b_encoder(
    checkpoint,
):
    """
    Reconstruct the exact tokenizer used during training.
    """

    tokenizer_name = checkpoint[
        "tokenizer"
    ]

    if tokenizer_name == "single_nucleotide":

        return lambda sequence: torch.tensor(
            [
                BASE_ID[base]
                for base in sequence
            ],
            dtype=torch.long,
        )

    if tokenizer_name == "one_hot":

        return lambda sequence: torch.tensor(
            [
                ONE_HOT[base]
                for base in sequence
            ],
            dtype=torch.float32,
        )

    if tokenizer_name == "overlap_6mer":

        return lambda sequence: torch.tensor(
            [
                KMER_ID[
                    sequence[i:i+6]
                ]
                for i in range(
                    len(sequence) - 5
                )
            ],
            dtype=torch.long,
        )

    if tokenizer_name == "nonoverlap_6mer":

        return lambda sequence: torch.tensor(
            [
                KMER_ID[
                    sequence[i:i+6]
                ]
                for i in range(
                    0,
                    len(sequence) - 5,
                    6,
                )
            ],
            dtype=torch.long,
        )

    if tokenizer_name == "bpe":

        if (
            "bpe_rules" not in checkpoint
            or "bpe_vocab" not in checkpoint
        ):

            raise ValueError(
                "The BPE checkpoint does not contain "
                "its learned tokenizer state."
            )

        bpe = RestoredBPE(
            rules=checkpoint[
                "bpe_rules"
            ],
            vocab=checkpoint[
                "bpe_vocab"
            ],
        )

        return lambda sequence: torch.tensor(
            bpe.encode(
                sequence
            ),
            dtype=torch.long,
        )

    raise ValueError(
        f"Unknown tokenizer: {tokenizer_name}"
    )


def load_group_b(
    requested_tokenizer,
):
    """
    Rebuild the custom Transformer and load its learned weights.
    """

    tokenizer_name, internal_summary = (
        choose_group_b_tokenizer(
            requested_tokenizer
        )
    )

    checkpoint_path = (
        find_group_b_checkpoint(
            tokenizer_name
        )
    )

    checkpoint = load_checkpoint(
        checkpoint_path
    )

    model = DNAClassifier(
        input_kind=checkpoint[
            "input_kind"
        ],
        vocab_size=int(
            checkpoint[
                "vocab_size"
            ]
        ),
        max_length=int(
            checkpoint[
                "max_length"
            ]
        ),
        d_model=int(
            checkpoint[
                "d_model"
            ]
        ),
        nhead=int(
            checkpoint[
                "nhead"
            ]
        ),
        layers=int(
            checkpoint[
                "layers"
            ]
        ),
        dropout=float(
            checkpoint[
                "dropout"
            ]
        ),
    )

    model.load_state_dict(
        checkpoint["state_dict"]
    )

    model = model.to(
        DEVICE
    )

    model.eval()

    encoder = build_group_b_encoder(
        checkpoint
    )

    information = {
        "group": "B",
        "family": "Custom Transformer",
        "tokenizer": tokenizer_name,
        "checkpoint": str(checkpoint_path),
        "internal_summary": internal_summary,
        "checkpoint_data": checkpoint,
    }

    return model, encoder, information


def predict_group_b(
    model,
    encoder,
    sequences,
    batch_size=64,
):
    """
    Tokenize new DNA exactly as training did and predict in batches.
    """

    probabilities = []

    for start in range(
        0,
        len(sequences),
        batch_size,
    ):

        batch_sequences = (
            sequences[
                start:start + batch_size
            ]
        )

        encoded = [
            encoder(sequence)
            for sequence
            in batch_sequences
        ]

        lengths = torch.tensor(
            [
                len(example)
                for example
                in encoded
            ],
            dtype=torch.long,
        )

        padding_value = (
            0.0
            if encoded[0].dtype.is_floating_point
            else 0
        )

        x = pad_sequence(
            encoded,
            batch_first=True,
            padding_value=padding_value,
        )

        positions = torch.arange(
            x.shape[1]
        ).unsqueeze(0)

        padding_mask = (
            positions
            >= lengths.unsqueeze(1)
        )

        with torch.no_grad():

            scores = model(
                x.to(DEVICE),
                padding_mask.to(
                    DEVICE
                ),
            )

            binding_probability = (
                torch.softmax(
                    scores,
                    dim=1,
                )[:, 1]
            )

        probabilities.extend(
            binding_probability
            .cpu()
            .tolist()
        )

    return probabilities


# =========================================================
# Shared model loader / prediction wrapper
# =========================================================

def load_student_model(
    group,
):
    """
    Load the trained model for the selected student pathway.
    """

    group = group.upper()

    if group == "A":
        return load_group_a()

    if group == "B":
        return load_group_b(
            CUSTOM_TOKENIZER
        )

    raise ValueError(
        "GROUP must be 'A' or 'B'."
    )


def predict_external_dna(
    group,
    model,
    tokenizer_or_encoder,
    information,
    sequences,
    batch_size=64,
):
    """
    Use the selected model family to predict external sequences.
    """

    group = group.upper()

    if group == "A":

        return predict_group_a(
            model=model,
            tokenizer=tokenizer_or_encoder,
            sequences=sequences,
            max_length=information[
                "max_length"
            ],
            batch_size=batch_size,
        )

    return predict_group_b(
        model=model,
        encoder=tokenizer_or_encoder,
        sequences=sequences,
        batch_size=batch_size,
    )

# 5. Load **your** trained model

### Function: `load_student_model(GROUP)`

This function does the appropriate work for your group.

For Group A:

```text
find DNABERT checkpoint
↓
rebuild DNABERT architecture
↓
load state_dict
↓
load local 6-mer tokenizer
```

For Group B:

```text
choose tokenizer
↓
find its checkpoint
↓
rebuild custom Transformer
↓
load state_dict
↓
reconstruct the same tokenizer
```

Notice that **loading a model means architecture + learned weights + preprocessing**.

In [ ]:
# ▶️ RUN — load the model trained in Notebook 3A or 3B

model, tokenizer_or_encoder, model_info = (
    load_student_model(
        GROUP
    )
)


print("Model family:", model_info["family"])
print("Tokenizer:", model_info["tokenizer"])
print("Checkpoint:")
print(model_info["checkpoint"])

## What is `state_dict`?

A PyTorch `state_dict` is a collection of learned tensors.

Conceptually:

```mermaid
flowchart LR
    A["Model architecture"] --> C["Usable trained model"]
    B["state_dict<br/>learned weights"] --> C
```

Rebuilding only the architecture would give us a new, untrained model.

Loading the `state_dict` restores what the students' training run learned.

# 6. Predict the external astrocyte DNA

### Function: `predict_external_dna(...)`

Inputs:

- the trained model,
- the exact tokenizer/preprocessing used during training,
- the new DNA sequences,
- `batch_size=64` → process 64 external sequences at a time.

Output:

- one **Binding probability** for every DNA sequence.

The model weights do not change.

In [ ]:
# ▶️ RUN — inference on ENCSR000AOO external DNA

external_sequences = (
    test_data["sequence"]
    .tolist()
)


binding_probabilities = (
    predict_external_dna(
        group=GROUP,
        model=model,
        tokenizer_or_encoder=tokenizer_or_encoder,
        information=model_info,
        sequences=external_sequences,
        batch_size=64,
    )
)


test_data["binding_probability"] = (
    binding_probabilities
)


# Threshold 0.5:
# probability >= 0.5 → Binding
# probability < 0.5  → Background
test_data["predicted_label"] = (
    test_data[
        "binding_probability"
    ]
    .ge(0.5)
    .astype(int)
)


test_data["prediction"] = (
    test_data["predicted_label"]
    .map({
        0: "Background",
        1: "Binding",
    })
)


test_data["correct"] = (
    test_data["predicted_label"]
    == test_data["true_label"]
)


print(
    "Predicted",
    len(test_data),
    "external sequences."
)

test_data.head()

### What does the 0.5 threshold mean?

The model produces a Binding probability between 0 and 1.

For this simple classifier:

```text
Binding probability < 0.5
→ predict Background

Binding probability ≥ 0.5
→ predict Binding
```

ROC and Precision–Recall curves later examine performance **across many thresholds**, not only 0.5.

# 7. Calculate external-test metrics

These are the same metrics used in the earlier notebooks, but now they are measured on **another cell type**.

This answers:

> How well did the K562-trained model generalize to the astrocyte external dataset?

In [ ]:
# ▶️ RUN — external metrics

true_labels = (
    test_data["true_label"]
    .to_numpy()
)

predicted_labels = (
    test_data["predicted_label"]
    .to_numpy()
)

probabilities = (
    test_data["binding_probability"]
    .to_numpy()
)


external_metrics = {
    "accuracy": accuracy_score(
        true_labels,
        predicted_labels,
    ),

    "precision": precision_score(
        true_labels,
        predicted_labels,
        zero_division=0,
    ),

    "recall": recall_score(
        true_labels,
        predicted_labels,
        zero_division=0,
    ),

    "f1": f1_score(
        true_labels,
        predicted_labels,
        zero_division=0,
    ),

    "auroc": roc_auc_score(
        true_labels,
        probabilities,
    ),

    "auprc": average_precision_score(
        true_labels,
        probabilities,
    ),
}


external_metrics_table = (
    pd.DataFrame([
        external_metrics
    ])
)


external_metrics_table.round(3)

# 8. Confusion matrix

The confusion matrix asks:

> **What kinds of mistakes did the external model make?**

```text
                    Predicted
                 Background  Binding
Actual
Background           TN        FP
Binding              FN        TP
```

If performance changes on astrocyte DNA, the confusion matrix can show *how* it changed.

In [ ]:
# ▶️ RUN — external confusion matrix

cm = confusion_matrix(
    true_labels,
    predicted_labels,
)


ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Background",
        "Binding",
    ],
).plot()


plt.title(
    f"{model_info['family']} — "
    "ENCSR000AOO external test"
)

plt.show()

# 9. ROC curve

ROC asks:

> Across different thresholds, how well does the model rank Binding DNA above Background DNA?

A random classifier is near the diagonal.

A stronger classifier bends toward the upper-left.

In [ ]:
# ▶️ RUN — external ROC curve

false_positive_rate, true_positive_rate, _ = (
    roc_curve(
        true_labels,
        probabilities,
    )
)


plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=(
        f"External AUROC = "
        f"{external_metrics['auroc']:.3f}"
    ),
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
)


plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ENCSR000AOO ROC curve"
)

plt.legend()
plt.show()

# 10. Precision–Recall curve

This focuses directly on:

- **Precision** → when we predict Binding, how trustworthy is that prediction?
- **Recall** → how many real Binding sequences did we find?

In [ ]:
# ▶️ RUN — external Precision–Recall curve

precision_values, recall_values, _ = (
    precision_recall_curve(
        true_labels,
        probabilities,
    )
)


plt.plot(
    recall_values,
    precision_values,
    label=(
        f"External AUPRC = "
        f"{external_metrics['auprc']:.3f}"
    ),
)


plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "ENCSR000AOO Precision–Recall curve"
)

plt.legend()
plt.show()

# 11. Look at the model's probabilities

A predicted class hides useful information.

These two predictions both count as Binding:

```text
0.51 → Binding
0.99 → Binding
```

but the model is much more confident in the second one.

So we also look at the **distribution of Binding probabilities** for the two true classes.

In [ ]:
# ▶️ RUN — probability distributions

background_probabilities = (
    test_data.loc[
        test_data["true_label"] == 0,
        "binding_probability",
    ]
)


binding_probabilities = (
    test_data.loc[
        test_data["true_label"] == 1,
        "binding_probability",
    ]
)


plt.hist(
    background_probabilities,
    bins=20,
    alpha=0.6,
    label="True Background",
)


plt.hist(
    binding_probabilities,
    bins=20,
    alpha=0.6,
    label="True Binding",
)


plt.axvline(
    0.5,
    linestyle="--",
    label="0.5 threshold",
)


plt.xlabel(
    "Predicted Binding probability"
)

plt.ylabel(
    "Number of sequences"
)

plt.title(
    "Does the model separate the external classes?"
)

plt.legend()
plt.show()

### How to interpret this graph

A strong separation would look approximately like:

```text
Background probabilities
mostly toward 0

Binding probabilities
mostly toward 1
```

Heavy overlap means the model has difficulty separating the two external classes.

# 12. Which individual sequences fooled the model?

A model metric summarizes hundreds of predictions.

Looking at individual errors reminds us that every point represents a real DNA sequence.

In [ ]:
# ▶️ RUN — inspect some mistakes

mistakes = (
    test_data[
        ~test_data["correct"]
    ]
    .copy()
)


print(
    "Incorrect external predictions:",
    len(mistakes),
)


columns_to_show = [
    column
    for column in [
        "id",
        "sequence",
        "true_label",
        "prediction",
        "binding_probability",
        "source",
        "cell_type",
    ]
    if column in mistakes.columns
]


mistakes[
    columns_to_show
].head(10)

# 13. Internal validation vs. external validation

This is one of the most important comparisons in the bootcamp.

```mermaid
flowchart TD
    A["K562 training data"] --> B["Train model"]
    B --> C["K562 validation"]
    B --> D["Astrocyte external test"]

    C --> E["Same biological context"]
    D --> F["Different cell type"]

    E --> G["Compare performance"]
    F --> G
```

If external performance is lower, that does **not automatically mean the model is bad**.

It may mean the external data come from a different distribution or biological context.

This is called **distribution shift**.

In [ ]:
# 🔒 HELPER — find the internal validation metrics from Notebook 3A / 3B

internal_auroc = None
internal_auprc = None


if GROUP.upper() == "A":

    summary = model_info.get(
        "internal_summary"
    )

    if summary is not None:

        internal_auroc = (
            summary.get(
                "final_val_auroc"
            )
        )

        internal_auprc = (
            summary.get(
                "final_val_auprc"
            )
        )


elif GROUP.upper() == "B":

    selected_tokenizer = (
        model_info["tokenizer"]
    )

    summary_path = (
        GROUP_B_RESULTS
        / f"{selected_tokenizer}_summary.json"
    )

    if summary_path.exists():

        with open(summary_path) as handle:
            summary = json.load(
                handle
            )

        internal_auroc = (
            summary.get(
                "final_val_auroc",
                summary.get(
                    "best_val_auroc"
                ),
            )
        )

        internal_auprc = (
            summary.get(
                "final_val_auprc"
            )
        )

In [ ]:
# ▶️ RUN — compare internal and external metrics

if (
    internal_auroc is not None
    and internal_auprc is not None
):

    generalization_table = pd.DataFrame({
        "dataset": [
            "K562 validation",
            "Astrocyte external",
        ],

        "AUROC": [
            internal_auroc,
            external_metrics[
                "auroc"
            ],
        ],

        "AUPRC": [
            internal_auprc,
            external_metrics[
                "auprc"
            ],
        ],
    })


    print(
        generalization_table.round(3)
    )


    generalization_table.set_index(
        "dataset"
    )[
        [
            "AUROC",
            "AUPRC",
        ]
    ].plot(
        kind="bar"
    )


    plt.ylim(0, 1)

    plt.ylabel(
        "Metric value"
    )

    plt.title(
        "Internal validation vs external generalization"
    )

    plt.xticks(
        rotation=0
    )

    plt.show()


else:

    print(
        "Internal validation metrics were not found. "
        "External metrics are still valid."
    )

# 14. Save the external predictions

Saving predictions makes the result reproducible and lets you compare Group A and Group B later.

The output includes the original external DNA plus:

```text
binding_probability
predicted_label
prediction
correct
```

In [ ]:
# ▶️ RUN — save your external result

OUTPUT_DIR = (
    PROJECT_DIR
    / "notebook4_results"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


safe_model_name = (
    model_info["family"]
    .lower()
    .replace(" ", "_")
)


safe_tokenizer = (
    model_info["tokenizer"]
    .lower()
    .replace(" ", "_")
    .replace("-", "_")
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / (
        f"group_{GROUP.upper()}_"
        f"{safe_model_name}_"
        f"{safe_tokenizer}_"
        "ENCSR000AOO_predictions.csv"
    )
)


test_data.to_csv(
    OUTPUT_PATH,
    index=False,
)


print(
    "Saved:"
)

print(
    OUTPUT_PATH
)

# ✅ Final bootcamp checkpoint

Explain the complete experiment:

```mermaid
flowchart LR
    A["K562 CTCF DNA"] --> B["Train model"]
    B --> C["Save checkpoint"]
    C --> D["Load checkpoint"]
    E["ENCSR000AOO<br/>astrocyte DNA"] --> F["Same preprocessing"]
    F --> D
    D --> G["Inference"]
    G --> H["Binding probabilities"]
    H --> I["Confusion matrix"]
    H --> J["ROC / PR"]
    H --> K["Internal vs external"]
```

Be able to answer:

1. **Why must we use the same tokenizer at inference time?**
2. **What is stored in a checkpoint?**
3. **What is the difference between training and inference?**
4. **Why do we call `model.eval()`?**
5. **Why do we use `torch.no_grad()`?**
6. **What does the 0.5 threshold do?**
7. **What does the confusion matrix reveal?**
8. **What do ROC and Precision–Recall evaluate?**
9. **Why might external astrocyte performance differ from K562 validation performance?**
10. **What would it mean if performance generalizes well to the new cell type?**

The final scientific lesson is:

> **A model is useful not only when it fits the data it trained on, but when we carefully test whether its learned patterns transfer to genuinely new data.**

# 🎨 Optional External-Validation Visualization Playground

This section is **optional**. Nothing below is required to finish the notebook.

Use it when you want to ask your own question about a variable you created earlier.

```mermaid
flowchart LR
    A["Choose a variable"] --> B["Choose a term / column"]
    B --> C{"What do you want to see?"}
    C -->|"Counts / categories"| D["Bar plot"]
    C -->|"Distribution"| E["Histogram"]
    C -->|"Change across epochs"| F["Line plot"]
    C -->|"Relationship between numbers"| G["Scatter plot"]
```

## Two plotting patterns to remember

### One term / column

```python
VARIABLE["TERM"].plot(kind="hist")
```

Read it as:

> From this variable, choose this term, then plot it.

### Two terms / columns

```python
VARIABLE.plot(
    x="TERM1",
    y="TERM2",
    kind="scatter",
)
```

Read it as:

> Use `TERM1` for the x-axis and `TERM2` for the y-axis.

Useful `kind=` choices:

| `kind` | Good for |
|---|---|
| `"bar"` | comparing categories |
| `"hist"` | seeing a distribution |
| `"line"` | following change across epochs |
| `"scatter"` | comparing two numerical values |
| `"box"` | comparing distributions between groups |

If you forget what terms exist inside a pandas table, run:

```python
VARIABLE.columns.tolist()
```

## Important variables from Notebook 4

| Variable | What it contains | Terms you can explore |
|---|---|---|
| `test_data` | every external ENCSR000AOO sequence plus the model prediction | `sequence`, `true_label`, `binding_probability`, `predicted_label`, `prediction`, `correct`, and source metadata such as `cell_type` when present |
| `external_metrics_table` | external classifier metrics | `accuracy`, `precision`, `recall`, `f1`, `auroc`, `auprc` |
| `mistakes` | only external examples predicted incorrectly | same prediction/source terms found in `test_data` |
| `generalization_table` | internal K562 vs. external astrocyte performance, when internal metrics are available | `dataset`, `AUROC`, `AUPRC` |
| `model_info` | information about which model/checkpoint/tokenizer was loaded | use `model_info.keys()` to inspect it; this is a dictionary rather than a pandas table |

### Questions you could visualize

- How confident was the model on external DNA?
- Are Binding and Background probability distributions separated?
- How many external examples were correct vs. incorrect?
- Did AUROC/AUPRC decrease from K562 validation to astrocytes?
- Which true class produced more errors?

In [ ]:
# ▶️ OPTIONAL — inspect the available terms

print("test_data terms:")
print(test_data.columns.tolist())

print("\nexternal_metrics_table terms:")
print(external_metrics_table.columns.tolist())

print("\nmistakes terms:")
print(mistakes.columns.tolist())

if "generalization_table" in globals():
    print("\ngeneralization_table terms:")
    print(generalization_table.columns.tolist())

print("\nmodel_info keys:")
print(list(model_info.keys()))

## Copy a visualization recipe and change the terms

### External prediction confidence

```python
test_data["binding_probability"].plot(
    kind="hist",
    bins=20,
)
```

### Correct vs. incorrect predictions

```python
test_data["correct"].value_counts().plot(kind="bar")
```

### Binding probability grouped by the true class

```python
test_data.boxplot(
    column="binding_probability",
    by="true_label",
)
```

Here:

```text
true_label = 0 → Background
true_label = 1 → Binding
```

### All external metrics

```python
external_metrics_table.T.plot(
    kind="bar",
    legend=False,
)
```

### Internal vs. external generalization

If `generalization_table` was created:

```python
generalization_table.plot(
    x="dataset",
    y=["AUROC", "AUPRC"],
    kind="bar",
)
```

### Look only at mistakes

```python
mistakes["true_label"].value_counts().plot(kind="bar")
```

This asks:

> Were more of the mistakes true Binding sequences or true Background sequences?

**Student challenge:** make one visualization that supports your answer to: “Did my model generalize to astrocyte CTCF DNA?”